# Figure S1

### Prepare DECOV dataset for benchmark

In [ ]:
%%bash
### download decov genomes
# wget https://zenodo.org/records/5173012/files/DEVoC.fasta?download=1 -O DEVoC.fasta

In [ ]:
### Ran UHVDB/mine on DEVoC genomes

### Compare UHGV to UHVDB classification

In [5]:
### Load UHVDB/mine results
import polars as pl

# count number of total input genomes from DEVoC
devoc_seq_ids = []
with open('DEVoC.fasta') as f:
    for line in f:
        if line.startswith('>'):
            devoc_seq_ids.append(line.strip().lstrip('>'))

# load UHVDB/mine's report and rename
uhvdb_hq_mine_report = (
    pl.read_csv('results/uhvdb/mine/local/r2025_06/devoc/uhvdb/virusfilter/devoc.uhvdb_virus_class.tsv.gz', separator='\t',
        columns=['seq_name', 'completeness', 'warnings', 'uhvdb_virus_classification'])
    .rename({'seq_name': 'original_id'})
    .with_columns([pl.col('original_id').str.split('|provirus').list[0]])
    .filter(
        (pl.col('completeness') >= 90) &
        (pl.col('uhvdb_virus_classification').is_in(['confident', 'uncertain'])) &
        (
            (pl.col('warnings').is_null()) |
            (~pl.col('warnings').str.contains('>1.5x'))
        )
    )
)

In [6]:
### Download UHGV metadata for DEVoC genomes
import polars as pl

# download uhgv metadata
# !wget https://portal.nersc.gov/UHGV/metadata/uhgv_metadata.tsv

uhgv_metadata = pl.read_csv('uhgv_metadata.tsv', separator='\t', ignore_errors=True,
    columns=['uhgv_genome', 'original_id', 'original_study_alias', 'viral_confidence', 'checkv_completeness'])

# filter metadata to DEVoC viruses
uhgv_hq_devoc_viruses = (
    uhgv_metadata
        .filter(
            (pl.col('original_study_alias') == 'DEVoC') &
            (pl.col('checkv_completeness') >= 90)
        )
)

# join UHGV and UHVDB mine results
uhgv_uhvdb_hq_merged = (
    uhgv_hq_devoc_viruses
        .join(uhvdb_hq_mine_report, on='original_id', how='full', coalesce=True)
)

In [14]:
### Display numbers from benchmark
# Total input sequences
print(f'Number of input DEVoC genomes: {len(devoc_seq_ids)}')

# Count number of sequences not HQ by either method
num_agree = len(set(devoc_seq_ids) - set(uhvdb_hq_mine_report['original_id'].to_list()) - set(uhgv_hq_devoc_viruses['original_id'].to_list()))
print("Number of sequences negative by both methods:",
    num_agree
)
print("Number negative in UHGV:", len(set(devoc_seq_ids) - set(uhgv_hq_devoc_viruses['original_id'].to_list())))
print("Number negative in UHVDB:", len(set(devoc_seq_ids) - set(uhvdb_hq_mine_report['original_id'].to_list())))

# identify number of HQ viruses classified as "confident" or "uncertain" correctly by UHVDB
num_disagree = 0
uhvdb_confident = 0
uhgv_confident = 0
uhvdb_uncertain = 0
uhgv_uncertain = 0

for row in uhgv_uhvdb_hq_merged.iter_rows(named=True):
    if (row['viral_confidence'] == 'Confident'):
        uhgv_confident += 1
    if (row['viral_confidence'] == 'Uncertain'):
        uhgv_uncertain += 1
    if (row['uhvdb_virus_classification'] == 'confident'):
        uhvdb_confident += 1
    if (row['uhvdb_virus_classification'] == 'uncertain'):
        uhvdb_uncertain += 1

    if (row['viral_confidence'] == 'Confident') and (row['uhvdb_virus_classification'] == 'confident'):
        num_agree += 1

    elif (row['viral_confidence'] == 'Uncertain') and (row['uhvdb_virus_classification'] == 'uncertain'):
        num_agree += 1

    else:
        continue

print("Number of confident UHVDB classification:", uhvdb_confident)
print("Number of confident UHGV classification:", uhgv_confident)
print("Number of uncertain UHVDB classification:", uhvdb_uncertain)
print("Number of uncertain UHGV classification:", uhgv_uncertain)
print("Proportion agreement:", num_agree / len(devoc_seq_ids))

Number of input DEVoC genomes: 12986
Number of sequences negative by both methods: 11763
Number negative in UHGV: 11891
Number negative in UHVDB: 11814
Number of confident UHVDB classification: 1162
Number of confident UHGV classification: 1080
Number of uncertain UHVDB classification: 10
Number of uncertain UHGV classification: 15
Proportion agreement: 0.9852918527645156
